# 04 - Demucs Source-Separation Investigation

Companion to `03_filter_ablation.ipynb`, kept light (helpers copied from 03,
not re-running the whole DSP ablation).

**Problem that triggered this:** running `model=demucs` on a file with talking
appeared to do *nothing* — output ~ input. The brief's hypothesis is that a
music source separator's **vocals** stem captures speech/crying interference and
the non-vocal residual carries breathing. Our wrapper drops `vocals` and keeps
`drums+bass+other`. If Demucs (MUSIC-trained) doesn't route speech into
`vocals`, dropping it removes nothing -> 'did nothing'.

**A quick sanity check on white noise already showed** Demucs dumps almost all
energy into `drums` and leaves `vocals` near-silent — i.e. it has no meaningful
mapping for non-music. This notebook tests that on REAL labeled talk/crying
segments by listening to **every stem individually**, not just the kept sum.

**Key question:** on a talking segment, where does the speech land across
`[drums, bass, other, vocals]`? If it's smeared into `other`/`drums` (not
isolated in `vocals`), the brief's hypothesis fails for pretrained music-Demucs
on auscultation — a legitimate, reportable negative result.

## 0. Setup (helpers copied from 03)

In [ ]:
import json
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from IPython.display import Audio, HTML, display

REPO_ROOT = Path('..').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.model import DemucsModel
from torchaudio.pipelines import HDEMUCS_HIGH_MUSDB_PLUS

DATA_ROOT = REPO_ROOT / 'data'
MR_DIR = DATA_ROOT / 'medical_records'
MR_WAVS = MR_DIR / 'ALL'
MR_JSON = MR_DIR / 'merged-1761225412230.json'

DEMUCS_SR = HDEMUCS_HIGH_MUSDB_PLUS.sample_rate  # 44100
DEMUCS_SOURCES = ['drums', 'bass', 'other', 'vocals']

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 120)
RANDOM_STATE = 7

In [ ]:
# --- segment table + slicing (copied from 02/03 so slices match) ---
def has_value(x):
    return x is not None and x != '' and not (isinstance(x, float) and math.isnan(x))


def load_segment_dataframe(json_path=MR_JSON, wav_dir=MR_WAVS):
    with open(json_path, encoding='utf-8-sig') as f:
        payload = json.load(f)
    rows = []
    for file_entry in payload['files']:
        filename = file_entry['filename']
        wav_path = wav_dir / filename
        for markup_idx, segment in enumerate(file_entry.get('markup', [])):
            row = {
                'segment_id': f'{filename}:{markup_idx}',
                'filename': filename,
                'wav_path': str(wav_path),
                'wav_exists': wav_path.exists(),
                'start': float(segment.get('start', 0.0) or 0.0),
                'length': float(segment.get('length', 0.0) or 0.0),
                'phase': segment.get('phase'),
                'pathology': segment.get('pathology'),
                'quality': segment.get('quality'),
                'noise_level': segment.get('noise_level'),
            }
            row['end'] = row['start'] + row['length']
            row['has_pathology'] = has_value(row['pathology'])
            rows.append(row)
    return pd.DataFrame(rows)


segments = load_segment_dataframe()
segments = segments.loc[segments['wav_exists']].reset_index(drop=True)
print(f'{len(segments)} segments from {segments.filename.nunique()} files')

In [ ]:
# --- audio read + helpers (from 03) ---
def read_segment(row, pad_sec=0.05, normalize=False):
    wav_path = Path(row['wav_path'])
    info = sf.info(str(wav_path))
    sr = info.samplerate
    start_sec = max(0.0, float(row['start']) - pad_sec)
    end_sec = min(float(info.frames) / sr, float(row['end']) + pad_sec)
    start_frame = int(round(start_sec * sr))
    frames = max(1, int(round((end_sec - start_sec) * sr)))
    audio, sr = sf.read(str(wav_path), start=start_frame, frames=frames, dtype='float32', always_2d=False)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    if normalize:
        peak = np.max(np.abs(audio)) if len(audio) else 0.0
        if peak > 0:
            audio = audio / peak * 0.95
    return audio, sr


def _norm(x):
    peak = np.max(np.abs(x)) if len(x) else 0.0
    return x / peak * 0.95 if peak > 0 else x


def resample_to(audio, old_sr, new_sr):
    if old_sr == new_sr:
        return audio
    import librosa  # librosa.resample preferred over torchaudio (quality)
    return librosa.resample(audio, orig_sr=old_sr, target_sr=new_sr)

In [ ]:
# --- load HDEMUCS once; expose raw per-stem separation ---
_demucs = DemucsModel(input_sample_rate=DEMUCS_SR)  # wraps HDEMUCS, our model
_raw = _demucs.model  # the underlying torchaudio HDemucs


@torch.no_grad()
def separate_stems(audio_mono, sr):
    """Run HDEMUCS on a 1-D mono signal at the segment's SR.

    Returns dict[stem -> 1-D mono np.array] at DEMUCS_SR (44.1k). Audio is
    resampled to 44.1k first (librosa), since that is the model's native rate.
    """
    y = resample_to(np.asarray(audio_mono, dtype=np.float32), sr, DEMUCS_SR)
    x = torch.from_numpy(y)[None, None, :].repeat(1, 2, 1)  # [1,2,T] stereo
    out = _raw(x)[0]  # [n_sources, 2, T]
    stems = {name: out[i].mean(0).cpu().numpy() for i, name in enumerate(DEMUCS_SOURCES)}
    return stems, DEMUCS_SR

In [ ]:
# --- the core view: original + every stem + kept-sum + dropped ---
def demucs_stems(row, keep=('drums', 'bass', 'other'), fmax=4000, normalize_audio=True):
    """Spectrogram + player for: original, each of the 4 stems, kept-sum, dropped-sum.

    Energy (% of input) and Delta-vs-input (dB) are annotated so you can see
    WHERE talk/crying lands and whether keep-sum differs from the input at all.
    """
    audio, sr = read_segment(row, pad_sec=0.05, normalize=False)
    stems, dsr = separate_stems(audio, sr)
    ref = resample_to(np.asarray(audio, dtype=np.float32), sr, dsr)
    in_energy = float(np.mean(ref ** 2)) + 1e-12
    in_rms = math.sqrt(in_energy)

    keep = list(keep)
    drop = [s for s in DEMUCS_SOURCES if s not in keep]
    kept_sum = sum(stems[s] for s in keep)
    dropped_sum = sum(stems[s] for s in drop) if drop else np.zeros_like(ref)

    panels = {'original': ref}
    for s in DEMUCS_SOURCES:
        panels[s] = stems[s]
    panels[f'KEEP={"+".join(keep)}'] = kept_sum
    panels[f'DROP={"+".join(drop)}'] = dropped_sum

    label = (f"{row['segment_id']} | quality={row.get('quality')} | "
             f"level={row.get('noise_level')} | pathology={row.get('pathology')} | "
             f"{row['length']:.2f}s | seg_sr={sr} -> demucs_sr={dsr}")
    display(HTML(f'<b>{label}</b>'))

    n = len(panels)
    fig, axes = plt.subplots(1, n, figsize=(2.6 * n, 3.0), squeeze=False)
    for ax, (name, y) in zip(axes[0], panels.items()):
        ax.specgram(y, Fs=dsr, NFFT=512, noverlap=384, cmap='magma')
        e_pct = 100.0 * float(np.mean(y ** 2)) / in_energy
        ax.set_title(f'{name}\n{e_pct:.0f}% E', fontsize=8)
        ax.set_ylim(0, min(fmax, dsr // 2))
        ax.label_outer()
    plt.tight_layout()
    plt.show()

    # how different is the kept-sum from the input? (the 'did nothing' check)
    m = min(len(kept_sum), len(ref))
    delta_rms = float(np.sqrt(np.mean((kept_sum[:m] - ref[:m]) ** 2)))
    delta_db = 20.0 * math.log10(delta_rms / in_rms + 1e-12)
    display(HTML(f'<b>KEEP vs original: Δ rms={delta_rms:.2e} ({delta_db:+.1f} dB)</b> '
                 f'<span style="color:gray">(very negative dB => keep-sum ≈ input => "did nothing")</span>'))

    def _ph(x):
        return Audio(_norm(x) if normalize_audio else x, rate=dsr)._repr_html_()

    cells = [f'<td><b>{n}</b><br>{_ph(y)}</td>' for n, y in panels.items()]
    # 4 players per row to keep it readable
    rows_html = ['<table>']
    for i in range(0, len(cells), 4):
        rows_html.append('<tr>' + ''.join(cells[i:i + 4]) + '</tr>')
    rows_html.append('</table>')
    display(HTML(''.join(rows_html)))


def sample_category(quality=None, noise_level=None, n=2, random_state=RANDOM_STATE):
    mask = pd.Series(True, index=segments.index)
    if quality is not None:
        mask &= segments['quality'] == quality
    if noise_level is not None:
        mask &= segments['noise_level'] == noise_level
    part = segments.loc[mask]
    return part.sample(min(n, len(part)), random_state=random_state) if len(part) else part

## 1. Where does TALK land? (the core test)

If the brief's hypothesis held, talk would concentrate in **vocals** (high % E
there, low elsewhere) and `DROP=vocals` would sound like the speech. If instead
talk is smeared into `other`/`drums` and `vocals` is near-empty, the kept-sum ≈
input (Δ very negative dB) — confirming Demucs-music doesn't separate auscultation
speech.

In [ ]:
for _, row in sample_category(quality='talk', n=20, random_state=RANDOM_STATE + 3).iterrows():
    demucs_stems(row)

Well it's far from perfect. But for some voices and their reverberations it works good (takes about 60+% of energy (mostly high pitched female). And the most interesting thing is that i encountered recording that didn't work for a command line run but there it works perfect (38% energy and sounds right). Now other things i've heard, out of 18 recordings:
- everything just gone to other (other stems show about 0% but not empty, really heavily correlate with bad quality of speach, noise corrupts it so bad you can't understand): 4
- distributed equally except vocal=0%: 2
- 50/50 bass and drum: 1
- almost all to vocal (~95+%): 3
- most to vocal (~75%): 4
- good separation: 4



## 2. Crying

Crying is the most 'vocal-like' interference — if anything triggers the `vocals`
stem, it's this. Worth seeing whether the hypothesis fares better here than talk.

In [ ]:
for _, row in sample_category(quality='crying', n=20, random_state=RANDOM_STATE + 1).iterrows():
    demucs_stems(row)

Well it's far from perfect. But for some voices and their reverberations it works good (takes about 60+% of energy (mostly high pitched female). And the most interesting thing is that i encountered recording that didn't work for a comand line run but there it works perfect (38% energy and sounds right). Now other things i've heard, out of 18 recordings:
- everything just gone to other (other stems show about 0% but not empty, really heavily correlate with bad quality of speach, noise corrupts it so bad you can't understand): 4
- distributed equally except vocal=0%: 2
- 50/50 bass and drum: 1
- almost all to vocal (~95+%): 3
- most to vocal (~75%): 4
- good separation: 4

For crying, out of 7 (mostly kids crying loud):
- 80% but it seems fair: 2
- almost all to voice: 4
- good: 2
oh i'm tired listening to toddlers... Seems like everything breakes when there is a poor quality of vocals, need to be careful i think bad cases are strong outliers.

## 3. Other strong-noise types for contrast

movement / cough — non-vocal interference. Just to map the stem behavior broadly.

In [ ]:
for quality in ['movement', 'cough']:
    part = sample_category(quality=quality, n=1, random_state=RANDOM_STATE + 1)
    if len(part) == 0:
        print(f'(no segments for quality={quality})')
        continue
    display(HTML(f'<h4>quality = {quality}</h4>'))
    for _, row in part.iterrows():
        demucs_stems(row)

## 4. Try alternative keep_stems

If the default keep=(drums,bass,other) ≈ input, maybe a different subset is the
useful denoiser. Listen to keeping only `other`, or only `vocals`, etc. — purely
empirical, see what actually preserves breathing vs removes noise.

In [ ]:
talk_row = sample_category(quality='talk', n=1, random_state=RANDOM_STATE + 4)
if len(talk_row):
    row = talk_row.iloc[0]
    for keep in [('other',), ('vocals',), ('drums', 'bass', 'other')]:
        display(HTML(f'<h4>keep = {keep}</h4>'))
        demucs_stems(row, keep=keep)
else:
    print('no talk segment')

## 5. Does Demucs SPOIL the target? (breathing / clean pathology)

The talk/crying slices almost never contain breathing, so they only test
*voice removal* — not whether Demucs **damages the signal we want to KEEP**.
This is the make-or-break check for using it as a denoiser.

On a clean breathing / pathology segment the *desired* behavior is:
- `vocals` ~ empty (no voice to grab),
- `DROP=vocals` ~ silent,
- `KEEP=drums+bass+other` ~ the input (breathing passes through untouched,
  KEEP-vs-input Δ dB close to 0 / not strongly negative meaning it's preserved).

If instead `vocals` steals breathing energy, or KEEP loses diagnostic content,
that's a real cost of applying Demucs broadly.

In [ ]:
# Clean pathology targets: has a pathology label AND not noisy.
# (broad_noisy isn't computed in nb04's lighter table, so approximate 'clean'
#  as quality in {None,'clean'} and noise_level in {None,'low'}.)
def _is_clean(row):
    q = row.get('quality'); nl = row.get('noise_level')
    q_ok = (not has_value(q)) or q == 'clean'
    nl_ok = (not has_value(nl)) or nl == 'low'
    return q_ok and nl_ok

clean_mask = segments['has_pathology'] & segments.apply(_is_clean, axis=1)
clean_path = segments.loc[clean_mask]
print(f'{len(clean_path)} clean pathology segments '
      f'(pathologies: {sorted(clean_path.pathology.dropna().unique())})')

for _, row in clean_path.sample(min(4, len(clean_path)), random_state=RANDOM_STATE).iterrows():
    demucs_stems(row)

In [ ]:
# Also spot-check across specific pathology classes if present
for path in ['wet wheezing', 'dry whistling wheezing', 'crepitus', 'hard breathing']:
    part = clean_path.loc[clean_path['pathology'] == path]
    if len(part) == 0:
        print(f'(no clean segment for pathology={path})')
        continue
    display(HTML(f'<h4>pathology = {path}</h4>'))
    demucs_stems(part.sample(1, random_state=RANDOM_STATE).iloc[0])

## Notes / takeaways

_Fill in after listening:_
- Does talk land in `vocals`, or smear into `other`/`drums`?
- KEEP vs original Δ dB on talk segments — is it really ~'did nothing'?
- Does crying fare any better than talk?
- Verdict on the brief's hypothesis (vocals=interference) for pretrained music-Demucs.
- If it fails: implications — Demucs needs fine-tuning on auscultation, or pick a
  different separator. (Feeds the Phase-3 decision point.)